In [17]:
!nvidia-smi

Thu May 14 09:15:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [18]:
%%writefile cuda_program.cu

#include <iostream>
#include <cuda_runtime.h>

using namespace std;

// VECTOR ADDITION KERNEL
__global__ void vectorAdd(int *a, int *b, int *c, int n) {

    int tid = threadIdx.x + blockIdx.x * blockDim.x;

    if (tid < n) {
        c[tid] = a[tid] + b[tid];
    }
}

// MATRIX MULTIPLICATION KERNEL
__global__ void matrixMultiplication(int *a, int *b, int *c, int n) {

    int row = threadIdx.y + blockIdx.y * blockDim.y;
    int col = threadIdx.x + blockIdx.x * blockDim.x;

    if (row < n && col < n) {

        int sum = 0;

        for (int k = 0; k < n; k++) {
            sum += a[row * n + k] * b[k * n + col];
        }

        c[row * n + col] = sum;
    }
}

int main() {

    int choice;

    cout << "===== CUDA PROGRAM =====" << endl;
    cout << "1. Vector Addition" << endl;
    cout << "2. Matrix Multiplication" << endl;
    cout << "Enter your choice: ";
    cin >> choice;

    // VECTOR ADDITION
    if (choice == 1) {

        int n;

        cout << endl;
        cout << "Enter size of vectors: ";
        cin >> n;

        int *a = new int[n];
        int *b = new int[n];
        int *c = new int[n];
        int *cpu = new int[n];

        cout << endl;
        cout << "Enter elements of first vector:" << endl;

        for (int i = 0; i < n; i++) {
            cin >> a[i];
        }

        cout << "Enter elements of second vector:" << endl;

        for (int i = 0; i < n; i++) {
            cin >> b[i];
        }

        // CPU Verification
        for (int i = 0; i < n; i++) {
            cpu[i] = a[i] + b[i];
        }

        int *d_a, *d_b, *d_c;

        int size = n * sizeof(int);

        cudaMalloc((void**)&d_a, size);
        cudaMalloc((void**)&d_b, size);
        cudaMalloc((void**)&d_c, size);

        cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
        cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

        int threads = 256;
        int blocks = (n + threads - 1) / threads;

        cudaEvent_t start, end;

        cudaEventCreate(&start);
        cudaEventCreate(&end);

        cudaEventRecord(start);

        vectorAdd<<<blocks, threads>>>(d_a, d_b, d_c, n);

        cudaEventRecord(end);
        cudaEventSynchronize(end);

        float time = 0;

        cudaEventElapsedTime(&time, start, end);

        cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

        int error = 0;

        for (int i = 0; i < n; i++) {
            error += cpu[i] - c[i];
        }

        cout << endl;
        cout << "Result of Vector Addition:" << endl;

        for (int i = 0; i < n; i++) {
            cout << c[i] << " ";
        }

        cout << endl;
        cout << "Error: " << error << endl;
        cout << "Time Elapsed: " << time << " ms" << endl;

        cudaFree(d_a);
        cudaFree(d_b);
        cudaFree(d_c);

        delete[] a;
        delete[] b;
        delete[] c;
        delete[] cpu;
    }

    // MATRIX MULTIPLICATION
    else if (choice == 2) {

        int n;

        cout << endl;
        cout << "Enter matrix size n: ";
        cin >> n;

        int total = n * n;
        int size = total * sizeof(int);

        int *a = new int[total];
        int *b = new int[total];
        int *c = new int[total];
        int *cpu = new int[total];

        cout << endl;
        cout << "Enter elements of first matrix:" << endl;

        for (int i = 0; i < total; i++) {
            cin >> a[i];
        }

        cout << "Enter elements of second matrix:" << endl;

        for (int i = 0; i < total; i++) {
            cin >> b[i];
        }

        int *d_a, *d_b, *d_c;

        cudaMalloc((void**)&d_a, size);
        cudaMalloc((void**)&d_b, size);
        cudaMalloc((void**)&d_c, size);

        cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
        cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

        dim3 threadsPerBlock(16, 16);

        dim3 blocksPerGrid(
            (n + threadsPerBlock.x - 1) / threadsPerBlock.x,
            (n + threadsPerBlock.y - 1) / threadsPerBlock.y
        );

        cudaEvent_t start, end;

        cudaEventCreate(&start);
        cudaEventCreate(&end);

        cudaEventRecord(start);

        matrixMultiplication<<<blocksPerGrid, threadsPerBlock>>>(
            d_a, d_b, d_c, n
        );

        cudaEventRecord(end);
        cudaEventSynchronize(end);

        float time = 0;

        cudaEventElapsedTime(&time, start, end);

        cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

        // CPU Verification
        for (int row = 0; row < n; row++) {

            for (int col = 0; col < n; col++) {

                int sum = 0;

                for (int k = 0; k < n; k++) {
                    sum += a[row * n + k] * b[k * n + col];
                }

                cpu[row * n + col] = sum;
            }
        }

        int error = 0;

        for (int i = 0; i < total; i++) {
            error += cpu[i] - c[i];
        }

        cout << endl;
        cout << "Result Matrix:" << endl;

        for (int row = 0; row < n; row++) {

            for (int col = 0; col < n; col++) {

                cout << c[row * n + col] << " ";
            }

            cout << endl;
        }

        cout << endl;
        cout << "Error: " << error << endl;
        cout << "Time Elapsed: " << time << " ms" << endl;

        cudaFree(d_a);
        cudaFree(d_b);
        cudaFree(d_c);

        delete[] a;
        delete[] b;
        delete[] c;
        delete[] cpu;
    }

    else {

        cout << "Invalid Choice!" << endl;
    }

    return 0;
}

Overwriting cuda_program.cu


In [19]:
!nvcc cuda_program.cu -o cuda_program

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [20]:
!printf "1\n5\n1 2 3 4 5\n10 20 30 40 50\n" | ./cuda_program



===== CUDA PROGRAM =====
1. Vector Addition
2. Matrix Multiplication
Enter your choice: 
Enter size of vectors: 
Enter elements of first vector:
Enter elements of second vector:

Result of Vector Addition:
11 22 33 44 55 
Error: 0
Time Elapsed: 0.220704 ms


In [21]:
!printf "2\n3\n1 2 3\n4 5 6\n7 8 9\n1 0 0\n0 1 0\n0 0 1\n" | ./cuda_program

===== CUDA PROGRAM =====
1. Vector Addition
2. Matrix Multiplication
Enter your choice: 
Enter matrix size n: 
Enter elements of first matrix:
Enter elements of second matrix:

Result Matrix:
1 2 3 
4 5 6 
7 8 9 

Error: 0
Time Elapsed: 0.22528 ms
